In [2]:
from __future__ import annotations

import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

import numpy as np

PROJECT = Path("/home/baiyu/LearnStageConstraints")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

assert "envs/segment" in str(Path(sys.executable)), (
    f"Expected conda segment interpreter, got {sys.executable}"
)
print("Python:", sys.executable)

from experiments.artifacts import write_json
from runners.run_benchmark import run_benchmark

live_run_dir = PROJECT / "outputs" / "map_balanced_pooled" / "BarClean" / "method_seed_000"
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
sweep_root = PROJECT / "outputs" / "diagnostics" / f"barclean_kappa_sweep_cp3_{stamp}"
backup_dir = sweep_root / "original_before_sweep"
sweep_root.mkdir(parents=True, exist_ok=False)

if not live_run_dir.is_dir():
    raise FileNotFoundError(f"Missing current run artifacts: {live_run_dir}")
shutil.copytree(live_run_dir, backup_dir)

cases = [
    ("auto", None),
    ("kappa8", 8.0),
    ("kappa15", 15.0),
]
print("Sweep root:", sweep_root)
print("Fixed cutpoint indices:", [0, 1, 3], "(only index 2 / Stage 3 endpoint is free)")


Python: /home/baiyu/miniforge3/envs/segment/bin/python
Sweep root: /home/baiyu/LearnStageConstraints/outputs/diagnostics/barclean_kappa_sweep_cp3_20260828_135608
Fixed cutpoint indices: [0, 1, 3] (only index 2 / Stage 3 endpoint is free)


In [3]:
summaries = {}
completed_cases = []
try:
    for label, kappa in cases:
        case_dir = sweep_root / label
        case_dir.mkdir(parents=True, exist_ok=False)
        print()
        print(f"=== Starting {label}: map_progress_kappa={kappa} ===", flush=True)
        summaries[label] = run_benchmark(
            methods=["map_balanced_pooled"],
            datasets=["BarClean"],
            method_seeds=[0],
            config_root=PROJECT / "configs",
            outdir=case_dir / "benchmark",
            method_overrides={
                "map_balanced_pooled": {
                    "fixed_true_cutpoint_indices": [0, 1, 3],
                    "map_progress_kappa": kappa,
                    "map_demo_num_workers": 1,
                    "disable_plots": True,
                }
            },
            refresh_demo_cache=False,
            resume=False,
        )
        archive_dir = case_dir / "artifacts"
        if not live_run_dir.is_dir():
            raise FileNotFoundError(f"Runner did not create expected artifacts: {live_run_dir}")
        shutil.move(str(live_run_dir), str(archive_dir))
        completed_cases.append(label)
        print(f"=== Completed {label}; archived at {archive_dir} ===", flush=True)
finally:
    # run_benchmark writes to the standard run directory. Preserve any failed partial
    # state separately, then restore the exact artifacts present before the sweep.
    if live_run_dir.exists():
        partial_dir = sweep_root / "partial_live_run"
        suffix = 1
        while partial_dir.exists():
            partial_dir = sweep_root / f"partial_live_run_{suffix}"
            suffix += 1
        shutil.move(str(live_run_dir), str(partial_dir))
        print("Preserved partial live artifacts at:", partial_dir)
    live_run_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(backup_dir, live_run_dir)
    print("Restored original standard output directory:", live_run_dir)

print("Completed cases:", completed_cases)



=== Starting auto: map_progress_kappa=None ===
[MAP] iter 000 | total=-1095.716 | constraint=-1095.716 | progress=0.000 | MeanAbsCutpointError=4.650 | MeanParameterError=0.024 | MeanParameterErrorRaw=0.022 | MeanStageSubgoalError=0.190 | PredictedConstraintCount=10.000 | SemanticConstraintF1=0.667 | SemanticConstraintMatchCount=7.000 | SemanticConstraintPrecision=0.700 | SemanticConstraintRecall=0.636 | TrueConstraintCount=11.000 | stage_ends=[[37, 60, 85, 102, 126], [42, 71, 101, 127, 152], [42, 67, 96, 114, 142], [44, 79, 108, 157, 179], [40, 67, 92, 146, 185]] | active=10 | progress_kappa=[1.3108, 4.5957, 0.4567, 1.5075, 2.3591]
[MAP] iter 001 | total=-442.184 | constraint=-94.946 | progress=-347.238 | MeanAbsCutpointError=4.850 | MeanParameterError=0.024 | MeanParameterErrorRaw=0.022 | MeanStageSubgoalError=0.199 | PredictedConstraintCount=10.000 | SemanticConstraintF1=0.667 | SemanticConstraintMatchCount=7.000 | SemanticConstraintPrecision=0.700 | SemanticConstraintRecall=0.636 |

In [4]:
def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

rows = []
details = {}
for label, requested_kappa in cases:
    artifact_dir = sweep_root / label / "artifacts"
    segmentation = load_json(artifact_dir / "segmentation.json")
    metrics = load_json(artifact_dir / "metrics.json")["scalar_metrics"]
    objectives = load_json(artifact_dir / "objectives.json")
    learned = load_json(artifact_dir / "learned_constraints.json")

    true_cuts = np.asarray(segmentation["true_cutpoints"], dtype=int)
    pred_cuts = np.asarray(segmentation["predicted_cutpoints"], dtype=int)
    third_delta = pred_cuts[:, 2] - true_cuts[:, 2]
    fixed_ok = bool(np.array_equal(pred_cuts[:, [0, 1, 3]], true_cuts[:, [0, 1, 3]]))

    modes = {
        (int(item["stage"]), item["feature_name"]): item
        for item in learned["feature_stage_modes"]
    }
    table_stage3 = modes[(2, "table_dist")]
    table_stage4 = modes[(3, "table_dist")]

    row = {
        "case": label,
        "requested_κ": "auto" if requested_kappa is None else requested_kappa,
        "cp3_Δ_per_demo": third_delta.tolist(),
        "cp3_mean_Δ": float(np.mean(third_delta)),
        "cp3_MAE": float(np.mean(np.abs(third_delta))),
        "all_fixed_cuts_exact": fixed_ok,
        "overall_cut_MAE": float(metrics["MeanAbsCutpointError"]),
        "semantic_F1": float(metrics["SemanticConstraintF1"]),
        "parameter_error_raw": float(metrics["MeanParameterErrorRaw"]),
        "objective": float(objectives["ModelObjectiveFinal"]),
        "Stage3_table_mode": table_stage3["mode"],
        "Stage3_table_eq_score": float(table_stage3["mode_scores"]["target_value"]),
        "Stage4_table_mode": table_stage4["mode"],
        "Stage4_table_eq_score": float(table_stage4["mode_scores"]["target_value"]),
    }
    rows.append(row)
    details[label] = {
        "requested_map_progress_kappa": requested_kappa,
        "fixed_true_cutpoint_indices": [0, 1, 3],
        "true_third_cutpoints": true_cuts[:, 2].tolist(),
        "predicted_third_cutpoints": pred_cuts[:, 2].tolist(),
        "third_cutpoint_delta": third_delta.tolist(),
        "summary": row,
        "stage3_modes": {
            feature: modes[(2, feature)]
            for feature in [item["name"] for item in learned["feature_schema"]]
        },
        "stage4_modes": {
            feature: modes[(3, feature)]
            for feature in [item["name"] for item in learned["feature_schema"]]
        },
    }

write_json(
    sweep_root / "comparison.json",
    {
        "cp3_definition": "third cutpoint / zero-based index 2 / Stage 3 endpoint",
        "cases": details,
    },
)

columns = [
    "case", "requested_κ", "cp3_Δ_per_demo", "cp3_mean_Δ", "cp3_MAE",
    "overall_cut_MAE", "semantic_F1", "parameter_error_raw", "objective",
    "Stage3_table_mode", "Stage3_table_eq_score",
    "Stage4_table_mode", "Stage4_table_eq_score",
]
print(" | ".join(columns))
for row in rows:
    print(" | ".join(str(row[column]) for column in columns))
print("Saved:", sweep_root / "comparison.json")


case | requested_κ | cp3_Δ_per_demo | cp3_mean_Δ | cp3_MAE | overall_cut_MAE | semantic_F1 | parameter_error_raw | objective | Stage3_table_mode | Stage3_table_eq_score | Stage4_table_mode | Stage4_table_eq_score
auto | auto | [-8, -12, -9, -35, -37] | -20.2 | 20.2 | 5.05 | 0.6666666666666666 | 0.022109567681802868 | -474.250009221465 | inactive | 9.423513467949164e-305 | lower_bound | 1.2997498377041015e-103
kappa8 | 8.0 | [-8, -12, -8, -35, -39] | -20.4 | 20.4 | 5.1 | 0.6666666666666666 | 0.02214380650749387 | 1015.592411397683 | inactive | 9.418716512879147e-305 | lower_bound | 1.7736565289017308e-125
kappa15 | 15.0 | [-18, -19, -19, -42, -44] | -28.4 | 28.4 | 7.1 | 0.5454545454545454 | 0.024542678570519363 | 3183.439576842876 | inactive | 2.444919351594001e-255 | lower_bound | 3.9325226815935666e-250
Saved: /home/baiyu/LearnStageConstraints/outputs/diagnostics/barclean_kappa_sweep_cp3_20260828_135608/comparison.json
